GridWorld

In [1]:
import numpy as np

In [3]:
class GridWorld:
    def __init__(self, stochastic=False, noise_prob=0.2):
        self.height = 4
        self.width = 4
        self.grid_size = (self.height, self.width)
        self.n_states = self.height * self.width
        self.n_actions = 4 # 0: up, 1: down, 2: left, 3: right
        self.stochastic = stochastic
        self.noise_prob = noise_prob
        self.goal_state = 15 # (3, 3)
        self.actions = {
            0: (-1, 0),
            1: (1, 0),
            2: (0, -1),
            3: (0, 1)
        }

    def coord_to_state(self, r, c):
        return r * self.width + c

    def state_to_coord(self, state):
        return divmod(state, self.width)

    def get_transitions(self, state, action):
        """Returns list of tuples: (probability, next_state, reward, is_terminal)"""
        if state == self.goal_state:
            return [(1.0, state, 0, True)]

        r, c = self.state_to_coord(state)

        def move(curr_r, curr_c, act):
            dr, dc = self.actions[act]
            nr, nc = max(0, min(self.height - 1, curr_r + dr)), max(0, min(self.width - 1, curr_c + dc))
            next_s = self.coord_to_state(nr, nc)
            reward = 10 if next_s == self.goal_state else -1
            return next_s, reward

        if not self.stochastic:
            next_s, reward = move(r, c, action)
            return [(1.0, next_s, reward, next_s == self.goal_state)]

        # if stochastic
        intended_ns, intended_reward = move(r, c, action)
        transitions = [(1.0 - self.noise_prob, intended_ns, intended_reward, intended_ns == self.goal_state)]
        other_actions = [a for a in range(4) if a != action]
        for alt_act in other_actions:
            next_s, reward = move(r, r, alt_act)
            transitions.append((self.noise_prob/3.0, next_s, reward, next_s == self.goal_state))

        return transitions